In [ ]:
# =============================================================================
# MAIN ORCHESTRATOR: Sync OHLCV -> Features -> Predictions -> View
# (Jupyter-friendly: clears previous cell output + plots at start of each cycle)
# =============================================================================

import time
import sqlite3
import pandas as pd
import sys
import os

# Jupyter helpers
from IPython.display import clear_output
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
import utils
from database_codes.sync_ohlcv  import sync_ohlcv
from database_codes.features    import sync_features
from database_codes.predictions import sync_predictions
from database_codes.pred_view   import display_predictions

# =============================================================================
# CONFIGURATION
# =============================================================================
POLL_SECONDS       = 60
SEPARATOR          = "=" * 80
INIT_START_DATE    = "2017-01-01 00:00:00"
LOOKBACK_MINUTES   = 240

# =============================================================================
# get_last_timestamp(db_path: str, table_name: str) -> str
# =============================================================================
def get_last_timestamp(db_path: str, table_name: str) -> str:
    try:
        with sqlite3.connect(db_path) as conn:
            result = pd.read_sql_query(
                f"SELECT MAX(open_time) as max_time FROM {table_name}",
                conn
            )
        max_time = result["max_time"].iloc[0]
        return max_time
    except Exception:
        return None

# =============================================================================
# truncate_log_if_configured(config: dict) -> None
# - If config["logging"]["log_file"] exists, truncate it (open with 'w')
# =============================================================================
def truncate_log_if_configured(config: dict) -> None:
    try:
        logging_cfg = config.get("logging", {})
        log_file = logging_cfg.get("log_file")
        if not log_file:
            return
        # If relative, make absolute relative to repo root
        if not os.path.isabs(log_file):
            try:
                repo_root = utils._repo_root()
                log_file = os.path.join(repo_root, log_file)
            except Exception:
                # fallback keep given path
                pass
        # Ensure directory exists
        log_dir = os.path.dirname(log_file)
        if log_dir and not os.path.exists(log_dir):
            try:
                os.makedirs(log_dir, exist_ok=True)
            except Exception:
                pass
        # Truncate file
        with open(log_file, "w", encoding="utf-8") as f:
            f.write("")
    except Exception:
        # Don't let log truncation crash the loop
        pass

# =============================================================================
# MAIN: Orchestration loop (Jupyter-friendly)
# - clear_output(wait=True) + plt.close("all") at start of each cycle so
#   previous prints & plots are removed and only current cycle output is visible.
# =============================================================================
def main_loop():
    config  = utils._load_config()
    db_path = config["database"]["db_path"]

    table_ohlcv = config["database"]["tables"]["ohlcv"]
    table_feat  = config["database"]["tables"]["features"]
    table_pred  = config["database"]["tables"]["predictions"]

    cycle = 1

    # Truncate once at startup (before any prints)
    truncate_log_if_configured(config)

    while True:
        # --- JUPYTER: clear previous cell output (text + plots)
        try:
            clear_output(wait=True)
        except Exception:
            # Not in Jupyter environment or clear failed -> ignore
            pass

        # Close all matplotlib figures so old plots don't persist
        try:
            plt.close("all")
        except Exception:
            pass

        # Truncate log file again at start of each cycle (optional)
        truncate_log_if_configured(config)

        # Now safe to print; only current cycle output will appear in the notebook cell
        print(f"\n{SEPARATOR}")
        print(f"🔄 Cycle #{cycle} at {utils.now_utc_str()}")
        print(SEPARATOR)

        # -----
        # OHLCV
        # -----
        print("📥 OHLCV SECTION")
        max_ohlcv = get_last_timestamp(db_path, table_ohlcv)

        if max_ohlcv:
            print(f"   Last: {max_ohlcv}")
            start_ms = int(pd.to_datetime(max_ohlcv).timestamp() * 1000) + 60000
        else:
            print(f"   Table empty. Initializing from {INIT_START_DATE}...")
            start_ms = int(pd.to_datetime(INIT_START_DATE).timestamp() * 1000)

        sync_ohlcv(start_ms)

        # --------
        # FEATURES
        # --------
        print("\n📊 FEATURES SECTION")
        max_feat      = get_last_timestamp(db_path, table_feat)
        max_ohlcv_now = get_last_timestamp(db_path, table_ohlcv)

        if max_ohlcv_now:
            if max_feat is None:
                start_feat = INIT_START_DATE
                print(f"   Table empty. Initializing from {start_feat}...")
                sync_features(start_feat)
            elif max_feat < max_ohlcv_now:
                start_feat = (pd.to_datetime(max_feat) + pd.Timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
                print(f"   Last: {max_feat}")
                sync_features(start_feat)
            else:
                print(f"   Last: {max_feat} (up to date)")
        else:
            print(f"   No OHLCV data yet. Skipping.")

        # -----------
        # PREDICTIONS
        # -----------
        print("\n🤖 PREDICTIONS SECTION")
        max_pred     = get_last_timestamp(db_path, table_pred)
        max_feat_now = get_last_timestamp(db_path, table_feat)

        if max_feat_now:
            if max_pred is None:
                start_pred = INIT_START_DATE
                print(f"   Table empty. Initializing from {start_pred}...")
                sync_predictions(start_pred)
            elif max_pred < max_feat_now:
                start_pred = (pd.to_datetime(max_pred) + pd.Timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
                print(f"   Last: {max_pred}")
                sync_predictions(start_pred)
            else:
                print(f"   Last: {max_pred} (up to date)")
        else:
            print(f"   No Features data yet. Skipping.")

        # ----
        # VIEW
        # ----
        print("\n📈 VIEW SECTION")
        # display_predictions should render inline plots in the notebook.
        # Ensure it uses matplotlib/plotly properly for inline display.
        display_predictions()

        # -----
        # WAIT
        # -----
        print(f"\n{SEPARATOR}")
        print(f"✅ Cycle #{cycle} complete. Sleeping {POLL_SECONDS}s...")
        print(SEPARATOR)

        cycle += 1
        time.sleep(POLL_SECONDS)

# =============================================================================
# ENTRYPOINT
# =============================================================================
if __name__ == "__main__":
    main_loop()


🔄 Cycle #448 at 2026-01-24 18:19:54
📥 OHLCV SECTION
   Last: 2026-01-24 18:18:00
✅ Synced 1 klines into 'bchusdt_1m'

📊 FEATURES SECTION
